In [0]:
%python
dbutils.widgets.text("catalog","{{catalog}}")

In [0]:
%python
catalog = dbutils.widgets.get("catalog")

In [0]:
--Table Creation Only once for Historical
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog || '.gold.dim_customers')
USING DELTA
AS
SELECT customer_id, name, city, state, signup_date, created_at, updated_at
FROM IDENTIFIER(:catalog || '.silver.customers');

-- Incremental
MERGE INTO IDENTIFIER(:catalog || '.gold.dim_customers') AS target
USING IDENTIFIER(:catalog || '.silver.customers') AS source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *;

In [0]:
--Table Creation Only once for Historical
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog ||'.gold.dim_products')
USING DELTA
AS
SELECT product_id, product_name, category, price, created_at, updated_at
FROM IDENTIFIER(:catalog ||'.silver.products');

-- Incremental
MERGE INTO IDENTIFIER(:catalog ||'.gold.dim_products') AS target
USING IDENTIFIER(:catalog ||'.silver.products') AS source
ON target.product_id = source.product_id

WHEN MATCHED AND source.updated_at > target.updated_at THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;

In [0]:
-- Create ONCE (run only first time)
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog ||'.gold.fact_sales')
USING DELTA
PARTITIONED BY (order_month)
AS
SELECT
    oi.order_item_id,
    oi.order_id,
    o.customer_id,
    oi.product_id,
    c.name                                    AS customer_name,
    c.city,
    c.state,
    p.product_name,
    p.category,
    o.order_date,
    o.order_status,
    oi.quantity,
    oi.price                                  AS unit_price,
    (oi.quantity * oi.price)                  AS revenue,
    DATE_FORMAT(o.order_date, 'yyyy-MM')      AS order_month,
    DATE_FORMAT(o.order_date, 'yyyy')         AS order_year,
    DAYOFWEEK(o.order_date)                   AS day_of_week
FROM IDENTIFIER(:catalog ||'.silver.order_items') oi
LEFT JOIN IDENTIFIER(:catalog ||'.silver.orders')    o ON oi.order_id   = o.order_id
LEFT JOIN IDENTIFIER(:catalog ||'.silver.customers') c ON o.customer_id = c.customer_id
LEFT JOIN IDENTIFIER(:catalog ||'.silver.products')  p ON oi.product_id = p.product_id;


-- Every subsequent run → MERGE (history preserved)
MERGE INTO IDENTIFIER(:catalog ||'.gold.fact_sales') AS target
USING (
    SELECT
        oi.order_item_id,
        oi.order_id,
        o.customer_id,
        oi.product_id,
        c.name                                AS customer_name,
        c.city,
        c.state,
        p.product_name,
        p.category,
        o.order_date,
        o.order_status,
        oi.quantity,
        oi.price                              AS unit_price,
        (oi.quantity * oi.price)              AS revenue,
        DATE_FORMAT(o.order_date, 'yyyy-MM')  AS order_month,
        DATE_FORMAT(o.order_date, 'yyyy')     AS order_year,
        DAYOFWEEK(o.order_date)               AS day_of_week
    FROM IDENTIFIER(:catalog ||'.silver.order_items')  oi
    LEFT JOIN IDENTIFIER(:catalog ||'.silver.orders')    o ON oi.order_id   = o.order_id
    LEFT JOIN IDENTIFIER(:catalog ||'.silver.customers') c ON o.customer_id = c.customer_id
    LEFT JOIN IDENTIFIER(:catalog ||'.silver.products')   p ON oi.product_id = p.product_id
) AS source
ON target.order_item_id = source.order_item_id

WHEN MATCHED AND (
    target.quantity    != source.quantity  OR
    target.unit_price  != source.unit_price OR
    target.order_status != source.order_status
) THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

In [0]:
-- --Table Creation Only once for Historical
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog ||'.gold.agg_revenue_by_state')
USING DELTA
AS
SELECT
    c.state,
    SUM(revenue)               AS total_revenue,
    COUNT(DISTINCT order_id)   AS total_orders,
    COUNT(DISTINCT customer_id) AS total_customers
FROM IDENTIFIER(:catalog ||'.gold.fact_sales') f
JOIN IDENTIFIER(:catalog ||'.gold.dim_customers') c USING (customer_id)
GROUP BY c.state;

-- Incremental
INSERT OVERWRITE    IDENTIFIER(:catalog ||'.gold.agg_revenue_by_state')
SELECT
    c.state,
    SUM(revenue)               AS total_revenue,
    COUNT(DISTINCT order_id)   AS total_orders,
    COUNT(DISTINCT customer_id) AS total_customers
FROM IDENTIFIER(:catalog ||'.gold.fact_sales')  f
JOIN IDENTIFIER(:catalog ||'.gold.dim_customers') c USING (customer_id)
GROUP BY c.state;

In [0]:
CREATE OR REPLACE TABLE IDENTIFIER(:catalog ||'.gold.agg_top_products')
USING DELTA
AS
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(f.revenue)                AS total_revenue,
    SUM(f.quantity)               AS total_units_sold,
    COUNT(DISTINCT f.order_id)    AS total_orders,
    ROUND(AVG(f.unit_price), 2)   AS avg_unit_price
FROM IDENTIFIER(:catalog ||'.gold.fact_sales') f
JOIN IDENTIFIER(:catalog ||'.gold.dim_products') p ON f.product_id = p.product_id
GROUP BY p.product_id, p.product_name, p.category
ORDER BY total_revenue DESC;

INSERT OVERWRITE IDENTIFIER(:catalog ||'.gold.agg_top_products')
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(f.revenue)                AS total_revenue,
    SUM(f.quantity)               AS total_units_sold,
    COUNT(DISTINCT f.order_id)    AS total_orders,
    ROUND(AVG(f.unit_price), 2)   AS avg_unit_price
FROM IDENTIFIER(:catalog ||'.gold.fact_sales') f
JOIN IDENTIFIER(:catalog ||'.gold.dim_products') p ON f.product_id = p.product_id
GROUP BY p.product_id, p.product_name, p.category;


-- Monthly sales trend
CREATE OR REPLACE TABLE IDENTIFIER(:catalog ||'.gold.agg_monthly_sales')
USING DELTA
AS
SELECT
    order_month,
    order_year,
    SUM(revenue)                  AS monthly_revenue,
    SUM(quantity)                 AS units_sold,
    COUNT(DISTINCT order_id)      AS total_orders,
    COUNT(DISTINCT customer_id)   AS total_customers,
    ROUND(AVG(revenue), 2)        AS avg_order_value
FROM IDENTIFIER(:catalog ||'.gold.fact_sales')
GROUP BY order_month, order_year
ORDER BY order_month;

INSERT OVERWRITE IDENTIFIER(:catalog ||'.gold.agg_monthly_sales')
SELECT
    order_month,
    order_year,
    SUM(revenue)                  AS monthly_revenue,
    SUM(quantity)                 AS units_sold,
    COUNT(DISTINCT order_id)      AS total_orders,
    COUNT(DISTINCT customer_id)   AS total_customers,
    ROUND(AVG(revenue), 2)        AS avg_order_value
FROM IDENTIFIER(:catalog ||'.gold.fact_sales')
GROUP BY order_month, order_year;

In [0]:
-- Daily sales trend
CREATE OR REPLACE TABLE IDENTIFIER(:catalog ||'.gold.agg_daily_sales')
USING DELTA
AS
SELECT
    order_date,
    order_month,
    SUM(revenue)                  AS daily_revenue,
    SUM(quantity)                 AS units_sold,
    COUNT(DISTINCT order_id)      AS total_orders,
    COUNT(DISTINCT customer_id)   AS total_customers
FROM IDENTIFIER(:catalog ||'.gold.fact_sales')
GROUP BY order_date, order_month
ORDER BY order_date;

INSERT OVERWRITE IDENTIFIER(:catalog ||'.gold.agg_daily_sales')
SELECT
    order_date,
    order_month,
    SUM(revenue)                  AS daily_revenue,
    SUM(quantity)                 AS units_sold,
    COUNT(DISTINCT order_id)      AS total_orders,
    COUNT(DISTINCT customer_id)   AS total_customers
FROM IDENTIFIER(:catalog ||'.gold.fact_sales')
GROUP BY order_date, order_month;


-- Revenue by category
CREATE OR REPLACE TABLE IDENTIFIER(:catalog ||'.gold.agg_revenue_by_category')
USING DELTA
AS
SELECT
    p.category,
    SUM(f.revenue)                AS total_revenue,
    SUM(f.quantity)               AS total_units_sold,
    COUNT(DISTINCT f.order_id)    AS total_orders,
    COUNT(DISTINCT f.customer_id) AS total_customers
FROM IDENTIFIER(:catalog ||'.gold.fact_sales') f
JOIN IDENTIFIER(:catalog ||'.gold.dim_products') p ON f.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;

In [0]:
OPTIMIZE dev.gold.fact_sales
ZORDER BY (order_date, customer_id, product_id);

-- OPTIMIZE dims
OPTIMIZE dev.gold.dim_customers
ZORDER BY (customer_id);

OPTIMIZE dev.gold.dim_products
ZORDER BY (product_id);

-- VACUUM (remove files older than 7 days)
VACUUM dev.gold.fact_sales        RETAIN 168 HOURS;
VACUUM dev.gold.dim_customers     RETAIN 168 HOURS;
VACUUM dev.gold.dim_products      RETAIN 168 HOURS;
VACUUM dev.gold.agg_monthly_sales RETAIN 168 HOURS;
VACUUM dev.gold.agg_top_products  RETAIN 168 HOURS;